# Project Dhwani — phase 1: compositional temporal audio grounding

**Task.** Given a recording and a query carrying a temporal condition, return *every* interval that satisfies it, or an empty list if none does.

| Type | Example | Answer |
|---|---|---|
| `PLAIN` | every dog bark | all bark intervals |
| `ORDINAL` | the second dog bark | one interval, or none |
| `AFTER` / `BEFORE` | every dog bark after the car horn | the barks that qualify |
| `NEXT_AFTER` | the first dog bark after the car horn | one interval |
| `WHILE` | every dog bark while music plays | barks overlapping music |
| `NOT_FOLLOWED` | every dog bark not followed by footsteps within 3 s | a subset of barks |
| `ABSENT` | every cat meow, when none occurs | `[]` |

Ground truth comes from deterministic predicates over an exact event timeline, so there is no annotation noise.

**What phase 1 decides.** Run open audio LLMs zero-shot and compare their profile against two diagnostic mocks:

- `mock:ignore_condition` returns every occurrence of the target sound, ignoring the condition.
- `mock:first_only` returns just the first occurrence, the failure TAG-Bench reported across 21 models.

If a real model scores well on `PLAIN` but tracks one of these mocks on the conditional types, the bottleneck is the decoder's handling of the condition, and **LoRA fine-tuning on composed queries is the right phase 2**. If it fails `PLAIN` as well, perception is the bottleneck and the method has to change.

**Runtime.** Cells 1 to 4 need no GPU. The model cells do: A100 is comfortable, a T4 loads in 4-bit automatically.

In [ ]:
# 1. Environment
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none (CPU runtime - cells 2 to 4 still work)")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
!pip install -q scipy soundfile librosa pyyaml pytest 2>&1 | tail -2
print("deps ready")

In [ ]:
# 2. Get the code and check it
import os, shutil
if os.path.exists("/content/mtech-thesis-project"):
    shutil.rmtree("/content/mtech-thesis-project")
%cd /content
!git clone -q https://github.com/Ani2512/mtech-thesis-project.git
%cd /content/mtech-thesis-project
!python -m pytest tests -q -p no:warnings

In [ ]:
# 3. Harness sanity check: procedural benchmark + diagnostic mocks
# (no GPU, no downloads, about 1 minute)
!python -m ctag.build_benchmark --source procedural --n-clips 100 --out data/proc

for mode in ["oracle", "ignore_condition", "first_only"]:
    print("\n" + "=" * 110)
    print("mock:" + mode)
    print("=" * 110)
    !python -m ctag.run_zeroshot --model mock:{mode} --bench data/proc/benchmark.jsonl --out runs/proc/mock_{mode}

### Reading the mock table

`oracle` should be 1.000 everywhere. Anything else means the harness is broken.

`ignore_condition` and `first_only` are the two failure signatures. Note where they differ: `ignore_condition` keeps `PLAIN` perfect but drops `ORDINAL` count accuracy to zero, while `first_only` shows a high `under_report_rate` on the multi-answer types. Those contrasts are what let us classify a real model.

In [ ]:
# 4. Build the real benchmark from ESC-50 (downloads about 600 MB once, roughly 5 minutes)
!python -m ctag.build_benchmark --source esc50 --n-clips 300 --p-overlap 0.45 --out data/esc50 --esc50-root data/esc50_raw

# Run the mocks on THIS benchmark too. The reference curves depend on the
# distribution of answer counts, so the mocks must be scored on the same data
# as the models for the comparison in cell 7 to mean anything.
for mode in ["oracle", "ignore_condition", "first_only"]:
    !python -m ctag.run_zeroshot --model mock:{mode} --bench data/esc50/benchmark.jsonl --out runs/esc50/mock_{mode}
print("mock reference runs done on the ESC-50 benchmark")

import json, collections
n, empty = collections.Counter(), collections.Counter()
for line in open("data/esc50/benchmark.jsonl"):
    d = json.loads(line)
    n[d["qtype"]] += 1
    empty[d["qtype"]] += d["expects_empty"]
print("\nqueries by type (share that are rejection queries):")
for t in n:
    print(f"  {t:<14} {n[t]:5d}   {empty[t] / n[t]:.0%}")
print("  total", sum(n.values()))

# listen to one clip and read its queries
from IPython.display import Audio, display
first = json.loads(open("data/esc50/benchmark.jsonl").readline())
print("\nexample clip:", first["clip_id"])
for line in open("data/esc50/benchmark.jsonl"):
    d = json.loads(line)
    if d["clip_id"] != first["clip_id"]:
        break
    print(f'  {d["qtype"]:<14} {d["text"]:<62} -> {d["answer"]}')
display(Audio(first["audio"]))

In [ ]:
# 5. Qwen2-Audio-7B-Instruct, zero-shot
#
# Precision is chosen from your GPU: fp16 where the ~17 GB of weights fit,
# NF4 4-bit otherwise. A Colab T4 has ~15 GB, so it loads in 4-bit.
# Progress prints seconds-per-query every 10 queries, so you can see the rate
# in the first minute and stop early if it is too slow.
#
# n=150 is a first signal (about 10 clips, every condition type represented).
# Raise it once you know the rate; the full set is 4404 queries.
!pip install -q "transformers>=4.44" accelerate bitsandbytes 2>&1 | tail -1
!python -m ctag.run_zeroshot --model qwen2-audio --bench data/esc50/benchmark.jsonl --n 150 --out runs/esc50/qwen2_audio

In [ ]:
# 6. Qwen2.5-Omni-7B, zero-shot
#
# Needs a newer transformers plus qwen-omni-utils. Changing transformers
# version may require Runtime > Restart session; if it complains, restart and
# re-run cells 2 and 4 (the ESC-50 download is cached, so the rebuild is quick).
!pip install -q "transformers>=4.52" qwen-omni-utils accelerate bitsandbytes 2>&1 | tail -1
!python -m ctag.run_zeroshot --model qwen2.5-omni --bench data/esc50/benchmark.jsonl --n 150 --out runs/esc50/qwen25_omni

In [ ]:
# 7. Phase 1 result: the failure curve, and the phase 2 decision
import json, glob, os

by_bench = {}
for p in sorted(glob.glob("runs/**/summary.json", recursive=True)):
    s = json.loads(open(p).read())
    by_bench.setdefault(os.path.basename(os.path.dirname(os.path.dirname(p))), {})[s["model"]] = s["by_type"]

TYPES = ["PLAIN", "ORDINAL", "AFTER", "BEFORE", "NEXT_AFTER", "WHILE", "NOT_FOLLOWED", "ABSENT", "ALL"]
COND = ["ORDINAL", "AFTER", "BEFORE", "NEXT_AFTER", "WHILE", "NOT_FOLLOWED"]

def table(runs, metric):
    print("\n" + metric)
    print("-" * 141)
    print(f"{'model':<24}" + "".join(f"{t:>13}" for t in TYPES))
    for m in sorted(runs):
        row = ""
        for t in TYPES:
            v = runs[m].get(t, {}).get(metric)
            row += f"{v:>13.3f}" if isinstance(v, (int, float)) else f"{'-':>13}"
        print(f"{m:<24}" + row)

# The ESC-50 benchmark is the one that matters; procedural is the harness check.
bench = "esc50" if "esc50" in by_bench else (list(by_bench) or [None])[0]
if bench is None:
    print("No runs found. Run cell 3 and cell 4 first.")
else:
    runs = by_bench[bench]
    print(f"benchmark: {bench}   ({len(runs)} runs)")
    for metric in ["f1@0.5", "count_acc", "under_report_rate", "rejection_f1"]:
        table(runs, metric)

    real = [m for m in runs if not m.startswith("mock")]
    print("\n" + "=" * 141)
    print("PHASE 2 DECISION")
    print("=" * 141)
    if not real:
        print("No real-model run yet. Run cell 5 and/or 6 on a GPU runtime.")
    for m in real:
        by = runs[m]
        plain = by.get("PLAIN", {}).get("f1@0.5") or 0.0
        vals = [by[t]["f1@0.5"] for t in COND if by.get(t, {}).get("f1@0.5") is not None]
        cond_mean = sum(vals) / len(vals) if vals else 0.0
        under = by.get("ALL", {}).get("under_report_rate") or 0.0
        parse = by.get("ALL", {}).get("parse_fail_rate") or 0.0
        print(f"\n{m}:  PLAIN f1={plain:.3f}   conditional mean f1={cond_mean:.3f}"
              f"   under-report={under:.0%}   parse-fail={parse:.0%}")
        # how closely does it track each diagnostic mock across the conditional types?
        for mock in ["mock:ignore_condition", "mock:first_only"]:
            if mock in runs:
                d = [abs(by[t]["f1@0.5"] - runs[mock][t]["f1@0.5"])
                     for t in COND
                     if by.get(t, {}).get("f1@0.5") is not None and runs[mock].get(t, {}).get("f1@0.5") is not None]
                if d:
                    print(f"    mean |gap| to {mock:<24} {sum(d)/len(d):.3f}")
        if parse > 0.3:
            print("  -> Output format is the first problem. Add constrained decoding before reading anything else.")
        elif plain < 0.25:
            print("  -> Fails even PLAIN grounding. PERCEPTION is the bottleneck, so LoRA on conditional")
            print("     queries will not fix it. Reconsider the method or switch backbone.")
        elif plain - cond_mean > 0.15:
            print("  -> Grounds the sound but drops the CONDITION. This is the decoder behaviour phase 2 targets.")
            print("     PROCEED: LoRA fine-tune on composed conditional queries, with the training-free")
            print("     decompose-and-combine agent as the comparison arm.")
            if under > 0.4:
                print("     High under-report rate too, the first_only signature. Cardinality-aware")
                print("     decoding belongs in the phase 2 ablation.")
        else:
            print("  -> Conditional performance tracks PLAIN. No condition-specific gap to exploit.")
            print("     Re-examine whether the queries are too easy before committing to phase 2.")

## Next

1. Save the per-type table and `runs/*/summary.json`. That table is the paper's Figure 1.
2. Hand-check about 20 rows of `runs/<model>/predictions.jsonl`, comparing `raw` to `pred`, to confirm the parser is not inventing failures.
3. Confirm on real recordings (DESED, TAG-Bench audio) before claiming the finding generalises beyond composed audio.
4. Phase 2 follows the decision printed above.